## Feature extraction functions

In [2]:
import pandas as pd

In [ ]:
def extract_deephase_score(seq, site):
    
    return score_df

In [ ]:
def extract_idr_achor_score(seq, site):
    
    return idr_score_df, achor_score_df

In [ ]:
def extract_compactness_score(idr_seq, site):
    
    return score_df

In [ ]:
def extract_disoflag_score(seq, site):
    
    return score_df

In [ ]:
def extract_phospholingo_score(seq, site):
    return score_df

In [ ]:
def extract_esm_embedding(seq, site):
    return protein_embedding_df, site_embedding_df

In [ ]:
def extract_ptmmamba_embedding(seq, site):
    return protein_embedding_df, site_embedding_df

In [ ]:
def extract_onehot_embedding(seq, site, left_flank=7, right_flank=7):
    return onehot_embedding_df

In [18]:
import os
import sys
import pandas as pd
import numpy as np
import src.files.iupred2a.iupred2a_lib as iupred2a_lib
from Bio import SeqIO
def extract_idr_seq(sequence, site):
    iupred_type = "long"
    # Predict disorder using iupred2a_lib
    iupred_scores = iupred2a_lib.iupred(sequence, iupred_type)[0]

    # Predict anchor regions using iupred2a_lib if anchor is enabled
    anchor_scores = iupred2a_lib.anchor2(sequence)[0]

    # Prepare the data for saving into a CSV file
    data = {
        "position": list(range(1, len(sequence) + 1)),
        "amino_acid": list(sequence),
        "iupred_score": iupred_scores,
        "anchor_score": anchor_scores,
    }

    # Create a DataFrame from the data
    df = pd.DataFrame(data)
    
    # Step 1: Add the 'call_idr' column
    df['call_idr'] = df['iupred_score'].apply(lambda x: 1 if x > 0.5 else 0)

    # Check if the provided site is within a '1' region
    if df.at[provided_site - 1, 'call_idr'] != 1:
        return None
    
    # Find the contiguous segments where call_idr == 1
    df['segment_id'] = (df['call_idr'] != df['call_idr'].shift()).cumsum()
    contiguous_segments = df[df['call_idr'] == 1].groupby('segment_id')

    for _, segment in contiguous_segments:
        # Check if the site falls within this segment
        if site in segment['position'].values:
            # Truncate the segment to a maximum length of 500 if necessary
            start_idx = segment.index[0]
            end_idx = segment.index[-1]

            if len(segment) > 500:
                # Ensure the site is within the truncated segment
                site_idx = df.index[df['position'] == site][0]
                if site_idx - start_idx > 250:
                    start_idx = max(site_idx - 250, start_idx)
                end_idx = min(start_idx + 500, end_idx)

            truncated_segment = df.loc[start_idx:end_idx]
            return "".join(truncated_segment['amino_acid'].tolist())

        return None

In [19]:
# Example usage
sequence = "MQVSTALVRCLTTLVALSYCRGHRQSGCGCAWREAVEWADTSEGSVIVR"
site = 15

region_sequence = extract_idr_seq(sequence, site)
print(region_sequence)

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LuVul\\1433predictor\\src\\files\\iupred2a/data/iupred2_long_energy_matrix'

In [ ]:
deephase_score_df = extract_deephase_score(seq, site)
idr_score_df, achor_score_df = extract_idr_achor_score(seq, site)
disoflag_score_df = extract_disoflag_score(seq, site)
phospholingo_score_df = extract_phospholingo_score(seq, site)

In [ ]:
protein_embedding_df, site_embedding_df = extract_esm_embedding(seq, site)
phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(seq, site)
onehot_embedding_df = extract_onehot_embedding(seq, site, left_flank, right_flank)

In [ ]:
compactness_score_df = extract_compactness_score(idr_seq, site)

In [ ]:
# merge feature dfs
plm_features_df = pd.concat([protein_embedding_df, site_embedding_df, phosphoprotein_embedding_df, phosphosite_embedding_df], axis=1)
bio_features_df = pd.concat([deephase_score_df, idr_score_df, achor_score_df, disoflag_score_df, phospholingo_score_df, onehot_embedding_df], axis=1)

In [ ]:
def mutate_seq(seq, *mutation, site):
    pass
    return mutated_seq

In [ ]:
def build_mutation_lst(mutation_string):
    pass
    return site, original_aa, mutated_aa

In [3]:
# build the features df:
curated_dataset_df = pd.read_csv("Table 1.csv")
curated_dataset_df = curated_dataset_df[curated_dataset_df["label"]==1]
curated_dataset_df

,human homology accession,human homology site without letter,human Site and Mutation,label,PMID
0,P43681,467,S467s,1,20141511
1,P11362,779,S779s,1,23564461
2,P98177,32,T32t,1,20141511
3,P14136,8,S8s,1,20141511
4,P08151,640,S640s,1,20141511
...,...,...,...,...,...
781,Q53ET0,368,S368s,1,18626018
782,Q05086-2,485,T485t,1,28835500
783,P40818,718,S718s/S716A,1,17720156
784,P40818,718,S718s,1,17720156


In [ ]:
# build the features df:
def build_features_df(uniprot, site, mutation_string):
    wt_seq = fetch_seq(uniprot)
    mut_lst = build_mutation_lst(mutation_string)
    if mut_lst:
        seq = mutate_seq(wt_seq, *mut_lst, site)
    else:
        seq = wt_seq
        
    deephase_score_df = extract_deephase_score(seq, site)
    idr_score_df, achor_score_df = extract_idr_achor_score(seq, site)
    disoflag_score_df = extract_disoflag_score(seq, site)
    phospholingo_score_df = extract_phospholingo_score(seq, site)
    protein_embedding_df, site_embedding_df = extract_esm_embedding(seq, site)
    phosphoprotein_embedding_df, phosphosite_embedding_df = extract_ptmmamba_embedding(seq, site)
    onehot_embedding_df = extract_onehot_embedding(seq, site)
    
    idr_seq = extract_idr_seq(seq, site)
    compactness_score_df = extract_compactness_score(idr_seq, site)
    
    # merge feature dfs
    plm_features_df = pd.concat([protein_embedding_df, site_embedding_df, phosphoprotein_embedding_df, phosphosite_embedding_df], axis=1)
    bio_features_df = pd.concat([deephase_score_df, idr_score_df, achor_score_df, disoflag_score_df, phospholingo_score_df, onehot_embedding_df], axis=1)
    
    return plm_features_df,bio_features_df


## Training dataset, independent dataset split

### 4.3	Feature selection 

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

### 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.

task 47: download and curate prediction data from clinvar

task 48: prediction

task 49: figure

task: model update